# Knee v6: DINO retry, learning-rate bracket on fold 0
First screen (lr_backbone 5e-5): dino16 0.741 macro, dinov2 0.673, ConvNeXt-T 0.833, and both DINO runs had a much higher TRAIN loss than ConvNeXt (0.75 / 0.79 vs 0.63), so they under-trained.
This run tests whether the learning rate is the reason: dino16 on both sides of 5e-5 (2e-5 on GPU 0, 2e-4 on GPU 1), everything else identical (fold 0, `--head tokens`, 12 epochs).
Reference fold-0 val macro (raw) for ConvNeXt-T by epoch: ep1 0.700 | ep3 0.763 | ep5 0.800 | ep9 0.828 | ep12 0.832. Old dino16 at 5e-5: ep5 0.710 | ep12 0.740.
EARLY STOP: if an arm is still below ~0.76 at epoch 5 it will not reach ConvNeXt: stop the session and save the quota.
Inputs: competition data, `knee-mri-pipeline` (patched knee_pipeline_v6), caching output, training output (oof_all.csv), the ConvNeXt 5-fold output. GPU T4 x2, internet ON, Save & Run All. ~1.2 h wall.

In [1]:
import os, sys, json, subprocess, time, threading
from pathlib import Path
import numpy as np, pandas as pd

DINO16 = "vit_small_patch16_224.dino"
LR_BACKBONE = os.environ.get("KNEE_LR")            # e.g. "2e-5" to retry with a lower backbone LR; None = train.py default (5e-5)
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA   = Path(os.environ.get("KNEE_DATA",   "/kaggle/input/competitions/rsna-knee-abnormality-detection"))
CACHE  = Path(os.environ.get("KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"))
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
BASE   = Path(os.environ.get("KNEE_BASE",   "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"))
for name, p in dict(CODE=CODE, DATA=DATA, CACHE=CACHE, LABELS=LABELS, BASE=BASE).items():
    assert p.exists(), f"{name} not found: {p}"
for f in ("geom.py", "tests/smoke_v6.py"):
    assert (CODE / f).exists(), f"{f} missing: dataset does not hold knee_pipeline_v6"
print("backbone:", DINO16, "| arms: lr_backbone 2e-5 (GPU 0) and 2e-4 (GPU 1)")

backbone: vit_small_patch16_224.dino | arms: lr_backbone 2e-5 (GPU 0) and 2e-4 (GPU 1)


In [2]:
# tests: if any fails, the assert stops the notebook BEFORE training starts
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})\n{r.stdout[-800:]}")
    assert r.returncode == 0, f"{t} failed:\n{r.stderr[-1500:]}"
EXTRA = ["--head", "tokens"]
print("extra train args:", EXTRA)

== tests/test_geom.py (exit 0)
canonical orientation identical for 24 acquisition conventions
right/left mirror normalisation OK for 3 planes
laterality rules OK
ALL GEOM TESTS PASS

== tests/smoke_v6.py (exit 0)
head=series: forward/backward OK, absent-series masking OK, params 5.69M
head=tokens: forward/backward OK, absent-series masking OK, params 5.18M
backbone convnext_tiny.in12k_ft_in1k: OK | vit_size None | params 28.9M
backbone efficientnet_b2: OK | vit_size None | params 8.9M
backbone vit_small_patch14_dinov2.lvd142m: OK | vit_size 168 | params 22.5M
backbone vit_small_patch16_224.dino: OK | vit_size 160 | params 22.6M
backbone deit_tiny_patch16_224.fb_in1k: OK | vit_size 160 | params 6.4M
v5-style state_dict loads into head='series'
KneeDS geometry parity OK
ALL V6 SMOKE TESTS PASS

extra train args: ['--head', 'tokens']


In [3]:
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--epochs", "12", "--pretrained",
             "--workers", "2", "--train_slices", "16", "--bs", "4", "--accum", "2"]
JOBS = {0: ("v6dino16_lr2e5_f0", ["0"], ["--backbone", DINO16, "--lr_backbone", "2e-5"]),
        1: ("v6dino16_lr2e4_f0", ["0"], ["--backbone", DINO16, "--lr_backbone", "2e-4"])}
LOGS, RC = {}, {}
def run(gpu, name, folds, jx):
    out = W / name
    if (out / "oof.csv").exists():
        print(f"[{name}] oof.csv exists, skipped"); RC[name] = 0; return
    cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + EXTRA + jx + ["--only_folds"] + folds + ["--out", str(out)]
    LOGS[name] = W / f"{name}.log"
    print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
    with open(LOGS[name], "w") as fh:
        RC[name] = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), cwd=str(CODE)).wait()
    print(f"[{name}] exit code {RC[name]}", flush=True)
threads = [threading.Thread(target=run, args=(g, n, f, x)) for g, (n, f, x) in JOBS.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(300)
    last = {}
    for n, f in LOGS.items():
        ls = [l for l in f.read_text().splitlines() if "ep " in l] if f.exists() else []
        last[n] = ls[-1] if ls else "(none yet)"
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished | exit codes:", RC)
for n, rc in RC.items():
    if rc != 0:
        print(f"--- {n} FAILED, log tail ---"); print("\n".join(LOGS[n].read_text().splitlines()[-25:]))

[v6dino16_lr2e5_f0] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone vit_small_patch16_224.dino --lr_backbone 2e-5 --only_folds 0 --out /kaggle/working/v6dino16_lr2e5_f0
[v6dino16_lr2e4_f0] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone vit_small_patch16_224.dino --lr_backbone 2e-4 --only_folds 0 --out /kaggle/working/v6dino16_lr2e4_f0
[0.08 h] {
 "v6dino16_lr2e5_f0": "(non

In [4]:
# ---- fold-0 comparison on the FIXED old labels (confident: y>=0.9 or <=0.1) ----
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEY4 = LAB[:4]
base = pd.read_csv(BASE)
Y = {l: base[f"y_{l}"].values for l in LAB}
def auc_l(y, p):
    m = (y >= 0.9) | (y <= 0.1)
    return roc_auc_score((y[m] >= 0.9).astype(int), p[m]) if m.sum() > 20 and len({*(y[m] >= 0.9)}) == 2 else np.nan
PRED = {"baseline": base}
for name, folds, jx in JOBS.values():
    f = W / name / "oof.csv"
    if f.exists():
        d = pd.read_csv(f); assert len(d) == len(base), f"{name}: row count differs from baseline"
        PRED[name.replace("_f0", "")] = d
cx = [p for p in Path("/kaggle/input").rglob("oof_v6convnext.csv")]
if cx: PRED["convnext"] = pd.read_csv(cx[0]); print("convnext OOF found:", cx[0])
else: print("convnext OOF not attached: skipping the convnext comparison")
assert any(k not in ("baseline", "convnext") for k in PRED), "no DINO oof.csv found: the training jobs failed, read the log tails printed by the training cell"
ref = next(v for k, v in PRED.items() if k not in ("baseline", "convnext"))
mask = ref[f"p_{LAB[0]}"].notna().values            # fold-0 validation studies = the rows this screen predicted
print("fold-0 validation studies:", int(mask.sum()))
RK = {n: {l: rankdata(d.loc[mask, f"p_{l}"].values) / mask.sum() for l in LAB} for n, d in PRED.items()}
names = [n for n in RK if n not in ("baseline", "convnext")]
if "convnext" in RK:
    for n in names: RK[f"ens {n}+convnext"] = {l: (RK[n][l] + RK["convnext"][l]) / 2 for l in LAB}
T = pd.DataFrame({n: {l: auc_l(Y[l][mask], r[l]) for l in LAB} for n, r in RK.items()})
T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
print("\nFold 0 only (ranks within fold):\n", T.round(3).to_string())
print("\nreference from the ConvNeXt run: fold-0 raw macro 0.8328, baseline 0.7795. Seed noise ~0.01.")
print("rule: within ~0.01 of ConvNeXt (0.833) or better -> run the 5-fold with that lr | 0.01-0.03 below -> only if the ensemble beats ConvNeXt alone by >0.015 | more than 0.03 below -> drop DINO for good")

convnext OOF found: /kaggle/input/notebooks/theokorir/knee-v6-convnext-5fold/oof_v6convnext.csv
fold-0 validation studies: 882

Fold 0 only (ranks within fold):
                   baseline  v6dino16_lr2e5  v6dino16_lr2e4  convnext  ens v6dino16_lr2e5+convnext  ens v6dino16_lr2e4+convnext
ACL                  0.833           0.850           0.627     0.892                        0.887                        0.813
MCL                  0.713           0.763           0.654     0.805                        0.792                        0.759
Medial Meniscus      0.757           0.799           0.623     0.870                        0.849                        0.795
Lateral Meniscus     0.715           0.724           0.674     0.776                        0.757                        0.750
Medial OA            0.861           0.871           0.704     0.874                        0.878                        0.835
Lateral OA           0.749           0.745           0.678     0.767        